# Name That LLM | Verified Llama Asset Cache
Metadata and file transport only. No model computation.

Meta Llama3.1 8B weights, distributed publicly by NousResearch under the same Llama3.1 license/AUP. All selected advertised official/public file hashes match. The saved job independently verifies actual downloaded bytes before reuse. No HF authentication or credentials are used.

For the credited reproduction of Dhruv Tibarewal's scored nlm-v27 v1, not a new model or claimed score. License and attribution are preserved.


In [ ]:
# Only metadata, public-file I/O and hashing. No model import or execution.
import datetime, hashlib, json, os, shutil, time, urllib.request, urllib.error
from pathlib import Path

PROTOCOL = {'checked_at_utc': '2026-10-25T22:05:09.181405+00:00', 'official_repo': 'meta-llama/Llama-3.1-8B', 'official_revision': 'd04e592bb4f6aa9cfee91e2e20afa771667e1d4b', 'official_gated': 'manual', 'public_repo': 'NousResearch/Meta-Llama-3.1-8B', 'public_revision': '1f47e50cdbe801ad8a5174156ec3a0655108fb9f', 'public_gated': False, 'license': 'llama3.1', 'selected_files': [{'path': 'LICENSE', 'git_blob_sha1': 'a7c3ca16cee30425ed6ad841a809590f2bcbf290', 'sha256': None, 'bytes': None}, {'path': 'USE_POLICY.md', 'git_blob_sha1': '81ebb55902285e8dd5804ccf423d17ffb2a622ee', 'sha256': None, 'bytes': None}, {'path': 'config.json', 'git_blob_sha1': 'cccf055d6f8f210387a248c91dc40e0c7a4bafab', 'sha256': None, 'bytes': None}, {'path': 'generation_config.json', 'git_blob_sha1': 'fc9506438b7b55383dc04c0816561442324846c3', 'sha256': None, 'bytes': None}, {'path': 'model-00001-of-00004.safetensors', 'git_blob_sha1': 'a70cf544ea82b645a9084d345203e3b855c474af', 'sha256': 'f8b9704ab09cdeb097aa4a0a24bca96f906eec36bad63ab495bc21475058601b', 'bytes': 4976698672}, {'path': 'model-00002-of-00004.safetensors', 'git_blob_sha1': '440f54a28d3e777128cca95168dd93017e99d8ef', 'sha256': 'c28b25e7541751056ee126627e007f8d4288319733285e9f7b17b9ff6eb313f0', 'bytes': 4999802720}, {'path': 'model-00003-of-00004.safetensors', 'git_blob_sha1': 'b42588cf2677760fc6f6c39b19e55a31fca75e1c', 'sha256': 'd8e9504dd4e4a146d484c52a97584ec14dac92237c46b064934af67a85e7d383', 'bytes': 4915916176}, {'path': 'model-00004-of-00004.safetensors', 'git_blob_sha1': 'e614ce32dfec70990b7029fb392a318df1622054', 'sha256': 'e4486f35c040f683f7d790354f66c169c109eb9fa0954a4a35d7c458a108405d', 'bytes': 1168138808}, {'path': 'model.safetensors.index.json', 'git_blob_sha1': '0fd8120f1c6acddc268ebc2583058efaf699a771', 'sha256': None, 'bytes': None}, {'path': 'original/params.json', 'git_blob_sha1': '0df11e3fad3f9420c84418a4b57cc2e87564d91b', 'sha256': None, 'bytes': None}, {'path': 'original/tokenizer.model', 'git_blob_sha1': 'a097ce5a06fce0fa3d685a8cfb175cef243dfde9', 'sha256': '82e9d31979e92ab929cd544440f129d9ecd797b69e327f80f17e1c50d5551b55', 'bytes': 2183982}, {'path': 'special_tokens_map.json', 'git_blob_sha1': 'd8cd5076496dbe4be2320312abc10adc43097b81', 'sha256': None, 'bytes': None}, {'path': 'tokenizer.json', 'git_blob_sha1': 'f916e71031fa08f3c6ef1680a590c15b52d3cdd9', 'sha256': None, 'bytes': None}, {'path': 'tokenizer_config.json', 'git_blob_sha1': 'cb9ec25536e44d86778b10509d3e5bdca459a5cf', 'sha256': None, 'bytes': None}], 'all_selected_metadata_matches': True, 'weight_shards': 4, 'weight_bytes': 16060556376, 'actual_download_bytes_verification': 'PENDING', 'license_and_AUP_blobs_identical': True, 'HF_rejection': 'Observed official request rejected without displayed reason; no repeated request/auth bypass'}
ROOT = Path('/kaggle/working')
MODEL = ROOT / 'llama-3.1' / '8b'
MODEL.mkdir(parents=True, exist_ok=True)
started = time.monotonic()
deadline_seconds = 3600
CHUNK = 8 * 1024 * 1024
report_path = ROOT / 'llama_asset_verification.json'
records = []
report = {
    'status': 'RUNNING', 'model_compute_performed': False,
    'purpose': 'Exact public-file transport for credited nlm-v27 reproduction',
    'protocol': PROTOCOL, 'files': records,
    'started_utc': datetime.datetime.now(datetime.timezone.utc).isoformat(),
}

def save_report():
    report['elapsed_seconds'] = time.monotonic() - started
    temp = report_path.with_suffix('.tmp')
    temp.write_text(json.dumps(report, indent=2) + '\n')
    temp.replace(report_path)

def inspect_file(path, expected):
    size = path.stat().st_size
    if expected.get('bytes') is not None:
        assert size == expected['bytes'], ('size mismatch', expected['path'], size)
    sha = hashlib.sha256()
    git = hashlib.sha1(b'blob ' + str(size).encode() + b'\0')
    with path.open('rb') as stream:
        while True:
            block = stream.read(CHUNK)
            if not block:
                break
            sha.update(block)
            git.update(block)
    digest = sha.hexdigest()
    if expected.get('sha256'):
        assert digest == expected['sha256'], ('SHA256 mismatch', expected['path'])
    else:
        assert git.hexdigest() == expected['git_blob_sha1'], ('Git blob mismatch', expected['path'])
    return {'path': expected['path'], 'bytes': size, 'sha256': digest,
            'git_blob_sha1': git.hexdigest() if not expected.get('sha256') else None,
            'official_expected_hash_verified': True}

assert shutil.disk_usage(ROOT).free > PROTOCOL['weight_bytes'] + 600_000_000
assert PROTOCOL['license_and_AUP_blobs_identical']
save_report()
try:
    for expected in PROTOCOL['selected_files']:
        assert time.monotonic() - started < deadline_seconds, 'I/O budget reached'
        rel = expected['path']
        assert '..' not in Path(rel).parts and not Path(rel).is_absolute()
        path = MODEL / rel
        path.parent.mkdir(parents=True, exist_ok=True)
        if path.exists():
            record = inspect_file(path, expected)
            record['reused_existing_verified_file'] = True
        else:
            url = ('https://huggingface.co/' + PROTOCOL['public_repo'] + '/resolve/'
                   + PROTOCOL['public_revision'] + '/' + rel)
            partial = path.with_name(path.name + '.part')
            request = urllib.request.Request(url, headers={'User-Agent': 'NTL-verified-assets/1.0'})
            # Public endpoint, no cookies, HF token, password or auth bypass.
            print('DOWNLOAD', rel, flush=True)
            with urllib.request.urlopen(request, timeout=90) as response, partial.open('wb') as stream:
                while True:
                    block = response.read(CHUNK)
                    if not block:
                        break
                    stream.write(block)
                    assert time.monotonic() - started < deadline_seconds, 'I/O budget reached'
            record = inspect_file(partial, expected)
            partial.replace(path)
            record['reused_existing_verified_file'] = False
        records.append(record)
        save_report()
        print('VERIFIED', rel, record['bytes'], record['sha256'], flush=True)

    config = json.loads((MODEL / 'config.json').read_text())
    assert config['model_type'] == 'llama'
    assert config['hidden_size'] == 4096 and config['num_hidden_layers'] == 32
    assert config['vocab_size'] == 128256
    assert config.get('torch_dtype') == 'bfloat16'
    index = json.loads((MODEL / 'model.safetensors.index.json').read_text())
    shards = sorted(set(index['weight_map'].values()))
    assert len(shards) == 4
    assert all((MODEL / name).is_file() for name in shards)
    assert all(any(r['path'] == name and r['official_expected_hash_verified'] for r in records)
               for name in shards)
    report.update({
        'status': 'COMPLETE', 'all_downloaded_bytes_verified': True,
        'required_file_count': len(PROTOCOL['selected_files']),
        'verified_file_count': len(records), 'weight_shards': shards,
        'model_path': str(MODEL),
        'original_nlm_v27_mdir_parts': ['/llama-3.1/', '/8b/'],
        'config_guard': {k: config[k] for k in ['model_type', 'hidden_size', 'num_hidden_layers',
                                             'vocab_size', 'torch_dtype']},
        'total_saved_file_bytes': sum(r['bytes'] for r in records),
        'source_reproduction_code_unchanged': True,
        'host_metric_or_model_forward_performed': False,
    })
    assert report['total_saved_file_bytes'] < 20_000_000_000
    (ROOT / 'README.md').write_text(
        '# Built with Llama\n\n'
        'Exact Llama 3.1 8B public file cache for a credited reproduction of '
        'Dhruv Tibarewal\'s nlm-v27 v1 (script362013743).\n\n'
        'Original model: Meta Llama3.1; public transport: '
        + PROTOCOL['public_repo'] + '@' + PROTOCOL['public_revision'] + '.\n'
        'Every selected file has an identical advertised official Meta hash. '
        'Downloaded bytes are independently verified. LICENSE and USE_POLICY.md '
        'are retained in llama-3.1/8b/. No model computation is performed here.\n'
        'This cache alone establishes no competition score or novelty.\n')
    (MODEL / 'NOTICE').write_text(
        'Llama 3.1 is licensed under the Llama 3.1 Community License, '
        'Copyright © Meta Platforms, Inc. All Rights Reserved.\n')
    save_report()
    print(json.dumps({k: v for k, v in report.items() if k not in ['protocol', 'files']}, indent=2), flush=True)
except BaseException as exc:
    report['status'] = 'ERROR'
    report['error_type'] = type(exc).__name__
    report['error'] = str(exc)
    save_report()
    raise
